# METAR comparison to reanalysis products 

## Overview
This notebook explores how we might compare METAR products to gridded reanalysis data sets, like ERA-5, WRF, or **AORC**. We compare single a single site in a time series and look at the comparison in some interactive map-based visualizations. 

For this example, we compare METAR with [AORC](https://registry.opendata.aws/noaa-nws-aorc/) data, a high-resolution reanalysis product produced by NOAA for the continental U.S. and Alaska from 1979-2023. This dataset has only 8 meteorological variables so is relatively simple to understand. 

1. Accessing METAR data
2. Accessing AORC data
3. Comparing time series 
4. Interactive map-based visualizations

## Prerequisites

| Concepts | Importance | Notes |
| --- | --- | --- |
| [Pandas](https://pandas.pydata.org/) | Necessary | Familiarity with tabular data |
| ? | Necessary |  |
| [S3 file access](https://github.com/s3fs-fuse/s3fs-fuse) | Helpful | Understanding data access from a cloud-hosted bucket |

- **Time to learn**: 25 minutes 

---

## Imports 

In [ ]:
# basic data wrangling  
import pandas as pd
import geopandas as gpd
import numpy as np
import xarray as xr

# data access  
import duckdb
import s3fs

# utilities 
from datetime import datetime, timedelta
# import metpy.calc as mpcalc
# from metpy.units import units
import scipy.signal as signal

# visualization packages 
import matplotlib.pyplot as plt
from seaborn import set_theme
set_theme()
from lonboard import viz, Map, ScatterplotLayer, HeatmapLayer
from lonboard.view_state import MapViewState
from lonboard.basemap import MaplibreBasemap, CartoStyle

## Time series comparison 

### Get METAR data 
Data is hosted on [Dynamical](https://dynamical.org/). We choose a single station, a time range, and select only the variables that we want to compare. Since this example is written by a snow scientist from Boulder, CO, our examples will focus on the winter time in the Colorado Rocky Mountains.  

The METAR column names are weird, so we define some more descriptive names here. The full documentation for column names can be found [here](https://unidata.github.io/gempak/man/parm/apxA.html). 

In [ ]:
col_refs = {
    'tmpc': ['Temperature', '°C'], 
    'relh': ['Relative Humidity', '%'],
    'drct': ['Wind Direction', 'degrees from North'],
    'sknt': ['Wind Speed', 'knots'],
}

In [ ]:
def valid_date(date): 
    '''
    Helper function to check that the date range is valid for METAR and AORC.
    '''
    assert isinstance(date, datetime)
    assert date >= datetime(1979, 1, 1, 0), 'Date cannot be earlier than 1979' 
    assert date < datetime(2024, 1, 1, 0), 'Date cannot be more recent than 2023' 
    return date

In [ ]:
# define target! 
target = (39.8328, -104.6575)
target_code = 'DEN'
target = (39.6426, -106.9177)
target_code = 'EGE'
# TODO add "find_closest_metar" functionality, bounding box search, etc. 

time_start = valid_date(datetime(2021, 1, 1, 0))
time_end = valid_date(datetime(2021, 5, 31, 23))

target_vars = ['tmpc', 'relh', 'drct', 'sknt']

The Eagle-Vail airport is located just off i70 to the West of Denver. 

In [ ]:
gdf = gpd.GeoDataFrame(
    {"station": [target_code]},
    geometry=gpd.points_from_xy([target[1]], [target[0]]),
    crs="EPSG:4326"
)
layer = ScatterplotLayer.from_geopandas(gdf, get_radius=7, radius_units="pixels", get_fill_color=[255, 0, 0, 200])
co_view = MapViewState(longitude=-105.7189, latitude=39.0598, zoom=6.5)
labeled_basemap = MaplibreBasemap(style=CartoStyle.Voyager)

# Render the interactive map
Map(layers=[layer], 
    view_state=co_view,
    basemap=labeled_basemap)

We'll need different files if the time range spans multiple years. 

In [ ]:
metar_base = "https://data.source.coop/dynamical/asos-parquet"
metar_urls = [f"{metar_base}/year={y}/data.parquet" for y in range(time_start.year, time_end.year + 1)]
metar_urls

In [ ]:
df = duckdb.execute(f"""
    SELECT valid, longitude, latitude, station, name, country, {', '.join(target_vars)}
    FROM read_parquet($1, hive_partitioning=true)
    WHERE station = $2 and 
          valid between $3 and $4
    ORDER BY valid
""", [metar_urls, target_code, time_start, time_end]).fetchdf()

In [ ]:
# df = duckdb.execute(f"""
#     SELECT *
#     FROM read_parquet($1, hive_partitioning=true)
#     WHERE station = $2
#     ORDER BY valid
# """, [metar_urls, target_code]).fetchdf()

In [ ]:
df.columns

In [ ]:
gdf_metar = gpd.GeoDataFrame(
                df, 
                geometry=gpd.points_from_xy(df.longitude,
                                            df.latitude)
)
gdf_metar

### Get the AORC data for the target time and location. 

AORC data has 8 variables: 
1. **APCP_surface:** 
Accumulated precipitation at surface (mm) 
2. **TMP_2maboveground:**
Temperature at 2m above ground (K)
3. **SPFH_2maboveground:**
Specific humidity at 2m above ground (g/g)
4. **DLWRF_surface:**
Downward longwave radiation (W/m^2)
5. **DSWRF_surface:**
Downward shortwave radiation (W/m^2)
6. **PRES_surface:** 
Pressure at surface (Pa)
7. **UGRD_10maboveground:**
East-West wind speed (m/s) 
8. **VGRD_10maboveground:**
North-South wind speed (m/s)

We'll use `xarray` and `s3fs` to open the cloud-hosted dataset. The advantage of this is that we can quickly access very large amounts of data. `df_aorc` has a full year of data over the Western U.S. 

In [ ]:
aorc_base = "noaa-nws-aorc-v1-1-1km" 
aorc_urls = [f"s3://{aorc_base}/{y}.zarr" for y in range(time_start.year, time_end.year + 1)]
aorc_urls

AORC data is hosted in Zarr format by NOAA on a public S3 bucket. 

In [ ]:
s3_out = s3fs.S3FileSystem(anon=True)
fileset_aorc = [
    s3fs.S3Map(root=url, s3=s3_out, check=False)
    for url in aorc_urls
]
df_aorc = xr.open_mfdataset(fileset_aorc, engine='zarr')
df_aorc

In [ ]:
print(f"Single variable size: {df_aorc.APCP_surface.size / (1024*1024*1024):.2f} GB")

Before doing anything else, we want to get rid of all the extra data! We select the single grid cell that contains the target point and select out time range. 

In [ ]:
# select the target point and target time range 
df_aorc_pt = df_aorc.sel(
    latitude=target[0], 
    longitude=target[1],
    method="nearest"
)
df_aorc_pt = df_aorc_pt.sel(
    time=slice(time_start, time_end)
)
del df_aorc # clean up

In [ ]:
print(f"Single variable size: {df_aorc_pt.APCP_surface.size / (1024*1024*1024):.10f} GB")

In order to compare this to the METAR data, we'll need to do some conversions. Since the METAR units are generally more understandable, we'll convert to those. 

| Variable | METAR unit | AORC unit | 
| --- | ---| --- |
| Temperature | **degrees C** | degrees K |
| Humidity | **Relative humidity** | Specific humidity (g/g) |
| Wind | **Wind direction (degrees) and wind speed (knots)** | E-W/N-S wind speeds (m/s) |

In [ ]:
def aorc_to_metar_cols(df_aorc): 
    '''
    Helper function for aligning the AORC and METAR variables for conversion. 
    '''
    for col in target_vars: 
        if col == 'tmpc' : 
            print('Converting TMP_2maboveground in K to tmpc in C') 
            df_aorc['tmpc'] = df_aorc['TMP_2maboveground'] - 273
        if col == 'relh' : 
            print('Converting SPFH_2maboveground to relh (%)') 
            q = df_aorc['SPFH_2maboveground']
            tmp = df_aorc['tmpc']
            prs = df_aorc['PRES_surface']
            df_aorc['relh'] = spech_to_relh(q, tmp, prs)
        if col == 'drct' : 
            print('Converting UGRD_10maboveground and VGRD_10maboveground to wind direction (deg)') 
            dir_rad = np.arctan2(df_aorc['UGRD_10maboveground'], df_aorc['VGRD_10maboveground'])
            dir_deg = np.degrees(dir_rad) + 180.0
            df_aorc['drct'] = np.mod(dir_deg, 360.0)  
        if col == 'sknt' : 
            print('Converting UGRD_10maboveground and VGRD_10maboveground to wind speed (knots)') 
            speed_mps = np.hypot(df_aorc['UGRD_10maboveground'], df_aorc['VGRD_10maboveground'])
            df_aorc['sknt'] = speed_mps * 1.94384
    return df_aorc

def spech_to_relh(q, tmp, prs): 
    '''
    Helper function for translating specific humidity to relative humidity
    '''
    p_hpa = prs / 100.0
    w = q / (1.0 - q) # mixing ratio 
    # calculate saturation vapor pressure (es) in hPa via Bolton (1980)
    es = 6.112 * np.exp((17.67 * tmp) / (tmp + 243.5)) 

    epsilon = 0.62198
    ws = epsilon * es / (p_hpa - es) # saturation mixing ratio
    
    # calculate relatie humidity, bounded from 0 to 100% 
    rh_array = (w / ws) * 100.0
    rh_array = np.clip(rh_array, 0.0, 100.0)
    return rh_array

We run the conversions then ditch the extra columns. 

In [ ]:
df_aorc_pt = aorc_to_metar_cols(df_aorc_pt)
df_aorc_pt = df_aorc_pt[target_vars]

Now that the units are harmonized, we need to deal with the timestamps. The METAR data has `datetime\[us, UTC\]` format (microsecond precision in UTC timezone), while the AORC uses `datetime\[ns\]` (nanosecond precision in local timezones). We'll move everything to the local timezone.  

In [ ]:
# convert to pandas for better manipulation
gdf_aorc = df_aorc_pt.to_dataframe().reset_index()

# convert both dataframes to local timezone 
local_tz = 'America/Denver'  
gdf_aorc['time'] = pd.to_datetime(gdf_aorc['time']).dt.tz_localize('UTC').dt.tz_convert(local_tz)
gdf_metar['valid'] = pd.to_datetime(gdf_metar['valid']).dt.tz_convert('America/Denver')

# conver to a geodataframe 
gdf_aorc = gpd.GeoDataFrame(
    gdf_aorc, 
    geometry=gpd.points_from_xy(gdf_aorc.longitude, gdf_aorc.latitude),
    crs="EPSG:4326" 
)

In [ ]:
gdf_aorc

### Compare! 
Now that everything is matched up and in the same format, we can make some plots to compare the two datasets. 

Here, we just plot each variable against the other. 

In [ ]:
fig, axes = plt.subplots(len(target_vars), 1, figsize=(3 * len(target_vars), 10)) 

for i, v in enumerate(target_vars): 
    axes[i].set_title(col_refs[v][0])
    axes[i].set_ylabel(col_refs[v][1])
    axes[i].plot(gdf_metar['valid'], gdf_metar[v], label='METAR', alpha=0.7) 
    axes[i].plot(gdf_aorc['time'], gdf_aorc[v], label='AORC', alpha=0.7)

    axes[i].legend(loc='upper left')

plt.tight_layout()
plt.show()

That's a little messy. Let's try plotting with a median filter. 

A kernel size of 25 for the median filter means that the window size will be about 1 day. 

In [ ]:
fig, axes = plt.subplots(len(target_vars), 1, figsize=(3 * len(target_vars), 10)) 

for i, v in enumerate(target_vars): 
    axes[i].set_title(col_refs[v][0])
    axes[i].set_ylabel(col_refs[v][1])
    axes[i].plot(gdf_metar['valid'], signal.medfilt(gdf_metar[v], kernel_size=25), label='METAR', alpha=0.7) 
    axes[i].plot(gdf_aorc['time'], signal.medfilt(gdf_aorc[v], kernel_size=25), label='AORC', alpha=0.7)

    axes[i].legend(loc='upper left')

plt.tight_layout()
plt.show()

Now, lets plot just the different between each the values. The METAR data is subtracted from the AORC, so this difference is positive (blue) where the AORC is overestimating the METAR data and negative (red) where it is underestimating. 

In [ ]:
fig, axes = plt.subplots(len(target_vars), 1, figsize=(3 * len(target_vars), 10)) 

for i, v in enumerate(target_vars):
    metar_series = pd.Series(
        # gdf_metar[v],
        signal.medfilt(gdf_metar[v], kernel_size=25), 
        index=gdf_metar['valid']
    )
    aorc_series = pd.Series(
        # gdf_aorc[v],
        signal.medfilt(gdf_aorc[v], kernel_size=25), 
        index=gdf_aorc['time']
    )
    
    aorc_aligned = aorc_series.reindex(metar_series.index, method='nearest')
    
    diff = aorc_aligned - metar_series
    x_dates = diff.index.values
    y_values = diff.values

    axes[i].axhline(0, color='darkgray', linewidth=1.2, linestyle='--')

    axes[i].fill_between(x_dates, 0, y_values, where=(y_values > 0), 
                         color='blue', alpha=0.4, interpolate=True)
    axes[i].fill_between(x_dates, 0, y_values, where=(y_values < 0), 
                         color='red', alpha=0.4, interpolate=True)

    max_val = np.nanmax(np.abs(y_values))
    if max_val > 0:
        axes[i].set_ylim(-max_val * 1.1, max_val * 1.1) 

    axes[i].set_title(col_refs[v][0])
    axes[i].set_ylabel(col_refs[v][1])
    axes[i].grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()